<a href="https://colab.research.google.com/github/garykbrixi/minerva/blob/main/examples/notebooks/loci_viewer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Minerva loci viewer

Contact maps for a genomic locus from Minerva's three interaction heads: **base pairing** (RNA),
**repeat** (DNA) and **protein**.

There are two views. The **heads** view is a single forward pass. The **Jacobian fingerprint** is
slower and more detailed. The `ug27` and `twoayggay` examples ship with the package, or you can
upload your own GenBank file.

Use a GPU runtime (`Runtime` → `Change runtime type`), fill in the form, then `Runtime` → `Run all`.

In [ ]:
import importlib.util
if importlib.util.find_spec("minerva") is None:
    !pip install -q "minerva-dna[viz] @ git+https://github.com/garykbrixi/minerva.git"

In [ ]:
#@title Settings { display-mode: "form" }
example  = "ug27"                 #@param ["ug27", "twoayggay", "upload your own"]
renderer = "interactive (Bokeh)"  #@param ["interactive (Bokeh)", "publication (PDF)"]
head_set = "l2 (last-2)"          #@param ["l2 (last-2)", "l6 (last-6)"]
#@markdown The Jacobian fingerprint runs a forward pass per position and token, so it is off by default and capped at `jac_max_tokens`.
run_jacobian   = False  #@param {type:"boolean"}
jac_max_tokens = 384    #@param {type:"integer"}
#@markdown Window in token positions. Leave both at 0 for the example's default window. `record` picks the LOCUS in an uploaded file.
window_start = 0  #@param {type:"integer"}
window_end   = 0  #@param {type:"integer"}
record       = 0  #@param {type:"integer"}
MODEL = "gbrixi/minerva-mlm"  #@param {type:"string"}

## Load the model

In [ ]:
import matplotlib.pyplot as plt
import torch
from transformers import AutoTokenizer
from minerva import MinervaForMaskedLM

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# bf16 needs an Ampere or newer GPU. The Colab T4 is older, so it gets fp16.
if DEVICE == "cpu":
    DTYPE = torch.float32
else:
    DTYPE = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16

tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = MinervaForMaskedLM.from_pretrained(MODEL, torch_dtype=DTYPE).to(DEVICE).eval()
print(DEVICE, DTYPE, "| heads:", sorted(model.linear_heads))

## Load the locus

In [ ]:
from minerva.data import example_path, extract_and_tokenize_gb

# Default record and token window for each bundled example (None = whole locus).
PRESETS = {
    "ug27":      dict(record=2, window=(748, 1772)),
    "twoayggay": dict(record=0, window=None),
}

if example == "upload your own":
    from google.colab import files   # outside Colab, set gb_file to a path instead
    gb_file, preset = next(iter(files.upload())), dict(record=record, window=None)
else:
    gb_file, preset = example_path(example), PRESETS[example]

window = (window_start, window_end) if window_end > window_start else preset["window"]
locus = extract_and_tokenize_gb(gb_file, use_existing_translations=True)[preset["record"]]
sequence, name = locus["sequence"], locus["locus_name"][:40]
all_tokens = tokenizer.convert_ids_to_tokens(tokenizer.encode(sequence))
print(name, "|", len(all_tokens), "tokens | window:", window or "whole locus")

## Viewer

The interactive viewer zooms with the mouse wheel, pans on drag, shows position and value on
hover, and has one threshold slider per channel. It also writes a standalone `.html` that opens
offline. The publication renderer writes a 600 dpi PDF of the raw contact probabilities.

In [ ]:
from bokeh.io import output_notebook, show
from minerva.visualization import (bokeh_contact_viewer, save_bokeh_html, plot_publication_locus,
                                   publication_head_contacts_rgb, render_fingerprints)

def show_contacts(channels, tokens, title, tag, to_rgb, kind="heads", vmax=1.0, offset=0):
    """Show contacts with the chosen renderer. `to_rgb` is only called for the PDF."""
    if renderer.startswith("interactive"):
        layout = bokeh_contact_viewer(channels, tokens=tokens, title=title, vmax=vmax, genome_offset=offset)
        output_notebook(hide_banner=True)   # Colab isolates each cell's output, so load BokehJS per cell
        show(layout)
        print("saved", save_bokeh_html(layout, f"{tag}.html"))
    else:
        plot_publication_locus(to_rgb(), title=title, overlay_kind=kind, genome_offset=offset, save=f"{tag}.pdf")
        plt.show()
        print("saved", f"{tag}.pdf")

## Heads view

In [ ]:
suffix = "_l6" if head_set.startswith("l6") else ""
heads = [f"{h}{suffix}" for h in ("base_pairing", "repeat", "protein")]
span = dict(seed_start=window[0], seed_end=window[1]) if window else {}

with torch.no_grad():
    pred = model.predict_contacts(sequence=sequence, tokenizer=tokenizer, head_names=heads,
                                  return_dict=True, **span)["predictions"]
channels = {h.removesuffix(suffix): pred[h].float().cpu().numpy() for h in heads}
tokens = all_tokens[window[0]:window[1]] if window else all_tokens

show_contacts(channels, tokens, f"Heads {head_set} — {name}", "heads",
              to_rgb=lambda: publication_head_contacts_rgb(channels, tokens=tokens),
              offset=window[0] if window else 0)

## Jacobian fingerprint

Runs only when `run_jacobian` is on. Without a window it takes the `jac_max_tokens` tokens at the
centre of the locus.

In [ ]:
if run_jacobian:
    if window:
        start, end = window[0], min(window[1], window[0] + jac_max_tokens)
    else:
        mid = len(all_tokens) // 2
        start, end = max(0, mid - jac_max_tokens // 2), min(len(all_tokens), mid + jac_max_tokens // 2)

    vocab = tokenizer.get_vocab()
    amino_acids = list("ACDEFGHIKLMNPQRSTVWY")
    fp = model.get_fingerprints(
        sequence, tokenizer,
        nuc_token_ids=[vocab[c] for c in "atgc"], aa_token_ids=[vocab[c] for c in amino_acids],
        jac_aa_order=amino_acids, position_range=(start, end), max_batch_size=32,
        autocast_dtype=DTYPE if DEVICE == "cuda" else None)

    show_contacts(fp.channels, fp.tokens, f"Jacobian — {name} [{start}:{end}]", "jacobian",
                  to_rgb=lambda: render_fingerprints(fp, style="publication"),
                  kind="fingerprint", vmax=10.0, offset=start)

## Download

In [ ]:
import os, sys
if "google.colab" in sys.modules:
    from google.colab import files
    for name in ("heads", "jacobian"):
        for ext in ("html", "pdf"):
            if os.path.exists(f"{name}.{ext}"):
                files.download(f"{name}.{ext}")